# The regression

**Step 2 (this part):** decide exactly what the model measures, and write it down **before**
fitting it. **Step 3 (added later):** fit it and read the results.

Why write the rules first? If you fit a model, look at the result, then adjust it, it's easy to
keep adjusting until it says what you hoped. Fixing the choices in advance, including which
checks will be run and what result would count as support, removes that temptation. Every check
listed below gets reported in step 3, whatever it shows.

The reasoning behind each choice comes from `01_explore.ipynb`.

## The question

> Holding time of day, airline, flight type and wind constant, are flights into **SFO** more
> likely to arrive late than flights into **ATL, EWR, LAX and MIA**?

## The decisions

| # | Decision | Choice | Why | Rejected alternative |
|---|---|---|---|---|
| 1 | Outcome | **Late yes/no** (arrived 15+ min after schedule), so **logistic regression** | It's the standard airline and government definition of punctuality, so the result is easy to explain and compare. Delay *minutes* are dominated by early arrivals (median −12 min) and capped at 56 min, so a minutes model would mostly describe how early flights land | Linear regression on delay minutes |
| 2 | Pulls included | The **11 regular pulls**, 7 Sep – 1 Oct | Consistent collection, and every flight has weather. The 4–5 Sep manual pulls held nearly all daytime flights and the flights without weather | All pulls (check R6) |
| 3 | Flights included | Only flights **scheduled at least 2 hours before their pull** | A late flight scheduled shortly before the 09:00 UTC pull is still in the air, so the API never returns it: flights in that window show 1–5% late vs about 14% otherwise. LAX and MIA have the most such flights, so keeping them would make those airports look better and inflate SFO's gap. The schedule is set in advance, so this filter can't be influenced by how the airport performs | Keep them all (check R7) |
| 4 | Operators included | Scheduled airlines and cargo. **Private jets and air ambulances excluded** | They fly on demand, not to a published schedule, so "late" doesn't mean the same thing | — |
| 5 | Airport | Compared as **SFO vs the other four together** (primary), then **each airport vs SFO** (secondary) | One primary test answers the main question without the false-alarm risk of four separate tests | Only pairwise tests |
| 6 | Time of day | **Night** (scheduled 00:00–05:59) vs the rest | On the regular pulls there is essentially no morning and little afternoon, so finer slots can't be estimated | 4 slots |
| 7 | Airline | United (baseline), Delta, American, Alaska: the airlines with **75+ flights** in this sample, **fixed** for every check and for October. The rest become Other | **Alaska must be separate**: 55% of its flights in this sample go to SFO, so lumping it into Other would blur the SFO comparison. The cut-off of 75 flights was chosen *to keep Alaska* (it has 93). Check R8 tests a lower cut-off, so the choice of cut-off doesn't decide the answer | A 100-flight cut-off, which would drop Alaska |
| 8 | Flight type | **Cargo** yes/no | Cargo is rarely late and concentrated at MIA and LAX | — |
| 9 | Weather | **Wind speed** (mph) | The only weather column that varies at more than one airport and can be trusted (step 1, section 3) | Rain, visibility, the weather label |

**What "holding wind constant" means here.** SFO is far windier than the others (about 16 mph on
average vs 4 at ATL). With wind in the model, the SFO result is **SFO's effect beyond what its
wind explains**, closer to an operational effect. Check R1 drops wind to show the **total** SFO
effect. Both are reported.

**Deliberately left out:**
- **Departure delay.** It's one of the ways an airport makes flights late, so controlling for it
  would hide the effect being measured.
- **Rain, visibility, the weather label.** Step 1 showed each is either unreliable or confined to one airport.
- **Flight length.** Not expected to matter much (step 1 review), so it's a check (R2) rather than in the main model.

## How uncertainty is measured

Flights in the same API response (one pull at one airport, typically 25–50 flights after
codeshares are merged) share that evening's conditions: a storm, an air-traffic hold. They aren't
independent, so standard errors are **clustered by batch** (pull × airport).

That gives only 11 batches per airport, so any SFO comparison rests on about 22 batches. With that
few, clustered confidence intervals tend to come out **too narrow**, which leans toward false
positives. So every claim must also pass a **cluster bootstrap**: resample each airport's 11
batches with replacement, re-fit the model 999 times, and take the middle 95% of the results as
the confidence interval. Some resamples may fail to fit (night has only 12 late flights). Those
are skipped and counted; if more than 5% fail, the bootstrap is reported as unreliable.

## Checks, fixed in advance (all reported in step 3)

| Check | What changes | What it tests |
|---|---|---|
| R1 | Drop wind speed | The **total** SFO effect, including what wind explains |
| R2 | Add scheduled flight length | Is SFO's result really a long-haul effect? |
| R3 | Passenger flights only | Is the result driven by cargo? |
| R4 | Add an indicator for each pull | Do pull-wide conditions explain it? |
| R5 | Re-fit 11 times, leaving out one pull each time | Does any single evening drive the result? |
| R6 | Include the 4–5 Sep pulls (without wind, so compare with R1). The 2-hour filter isn't applied to those flights, since those pulls didn't run at 09:00. Clusters: departure date × airport | Does the regular-pull restriction change the answer? |
| R7 | Keep the flights scheduled within 2 hours of the pull | How much did the capture bias matter? |
| R8 | Airline cut-off of 40 flights instead of 75 | Does the cut-off choice matter? |

## What will count as a result

**Primary test:** SFO vs the other four airports together. **"SFO is later"** if its odds ratio is
above 1 and **both** 95% intervals (clustered and bootstrap) stay above 1.

**Secondary tests**, each airport vs SFO, with p-values **Holm-adjusted** for running four tests:
- **"SFO is later than X"**: X's odds ratio is below 1 with adjusted p < 0.05 and a bootstrap interval below 1.
- **"X is later than SFO"**: the mirror image.
- Otherwise **"not distinguishable from SFO"**.

**"Robust"** means the claim's interval still excludes 1 in R1–R4, and its direction holds in
every leave-one-out fit (R5) and in R6–R8.

**Expectation, written now so it can't be adjusted later:** from step 1, SFO should be later than
the others overall, clearly later than ATL and EWR, borderline against LAX, and not
distinguishable from MIA. Fixing the capture bias may move LAX and MIA, since both were
flattered by it. Whatever the model says gets reported as it is.

## An honest limit of this pre-registration

These hypotheses came from the same September data the model will use. Writing the rules down
first stops the model from being tuned toward an answer, but it doesn't make this a true test. **The
true test is October:** once about 10 new pulls have landed (around 21 October), step 3's code is
re-run unchanged on **pulls from 3 October onward** (the 1 October pull is already in this sample),
data that played no part in forming these hypotheses.

The **primary test** decides. If SFO is later there too, the result is confirmed. If not, it is
**not confirmed**. That could mean the September result was noise, or simply that one month of
data is too small to detect a real but modest effect. The two can't be told apart without more data.

## Building the model table

Everything below checks that the data can support the model as specified. Nothing is fitted yet.

In [1]:
import pandas as pd
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor

from flights_data import load_flights, prepare

pd.set_option("display.precision", 3)

flights = load_flights()
clean = prepare(flights)

Loaded fct_flight_events live from Snowflake
2,897 flights


**Erratum (added 4 October 2026, after step 3 was fitted).** Decision 2 always said the sample
is "the 11 regular pulls, 7 Sep – 1 Oct", but the code below had only a start date. Once the
3 October pull landed, a re-run would have pulled October flights into this sample. The code now
enforces the end date the text always stated (`LAST_REGULAR_PULL`). **This changes no result:** the
sample is the same 1,726 flights it was when step 3 was fitted. No other step 2 code or text has
changed since commit `de7f6ce`.

In [2]:
FIRST_REGULAR_PULL = "2026-09-07"
LAST_REGULAR_PULL = "2026-10-01"   # erratum, 4 Oct 2026: the end date decision 2 always stated
MIN_HOURS_BEFORE_PULL = 2

# Airlines with 75+ flights in this sample, fixed on 2 October 2026. The list stays the same in
# every check and in the October re-run, so a check never changes two things at once
# (only check R8 changes it deliberately).
MODEL_CARRIERS = ["United Airlines", "Delta Air Lines", "American Airlines", "Alaska Airlines"]

sample = clean[
    clean["pull_date"].between(FIRST_REGULAR_PULL, LAST_REGULAR_PULL)
    & (clean["hours_before_pull"] >= MIN_HOURS_BEFORE_PULL)
    & ~clean["is_non_scheduled"]
].copy()

sample["is_late"] = sample["is_delayed_arrival"].astype(int)
sample["is_night"] = (sample["arrival_hour"] < 6).astype(int)
sample["carrier_group"] = sample["carrier"].where(sample["carrier"].isin(MODEL_CARRIERS), "Other")

# The evidence for the fixed list: flights per airline in this sample.
print(sample["carrier"].value_counts().head(6))
sample["cargo"] = sample["is_cargo"].astype(int)
sample["is_sfo"] = sample["is_sfo"].astype(int)

# One batch = one API response: one pull at one airport. Flights in a batch share conditions.
sample["batch"] = sample["pull_date"].dt.strftime("%Y-%m-%d") + " " + sample["arrival_airport"]

carrier
United Airlines      484
Delta Air Lines      339
American Airlines    298
Alaska Airlines       93
Air Canada            72
Frontier              70
Name: count, dtype: int64


### The model, written out

These are the exact formulas step 3 will fit. The two differ only in how airport enters: the
primary model has a single SFO yes/no input; the secondary model has one input per airport,
with `Treatment("SFO")` making SFO the baseline. `Treatment("United Airlines")` makes United
the baseline airline in both.

In [3]:
CONTROLS = (
    'is_night'
    ' + C(carrier_group, Treatment("United Airlines"))'
    ' + cargo'
    ' + weather_wind_speed'
)
PRIMARY_FORMULA = "is_late ~ is_sfo + " + CONTROLS
SECONDARY_FORMULA = 'is_late ~ C(arrival_airport, Treatment("SFO")) + ' + CONTROLS
INPUTS = SECONDARY_FORMULA.split("~")[1]   # used by the checks below

print("Primary:  ", PRIMARY_FORMULA)
print("Secondary:", SECONDARY_FORMULA)

Primary:   is_late ~ is_sfo + is_night + C(carrier_group, Treatment("United Airlines")) + cargo + weather_wind_speed
Secondary: is_late ~ C(arrival_airport, Treatment("SFO")) + is_night + C(carrier_group, Treatment("United Airlines")) + cargo + weather_wind_speed


### Check 1: is there enough data?

In [4]:
number_of_inputs = dmatrix(INPUTS, sample).shape[1] - 1   # minus the intercept

pd.Series({
    "flights": len(sample),
    "late flights": sample["is_late"].sum(),
    "pulls": sample["pull_date"].nunique(),
    "batches (pull x airport)": sample["batch"].nunique(),
    "model inputs": number_of_inputs,
    "late flights per input": round(sample["is_late"].sum() / number_of_inputs, 1),
})

flights                     1726.0
late flights                 250.0
pulls                         11.0
batches (pull x airport)      55.0
model inputs                  11.0
late flights per input        22.7
dtype: float64

A common rule of thumb asks for at least 10 late flights per input. More is better.

### Check 2: does every group have flights and late flights?

A group with no late flights at all would break a logistic regression.

In [5]:
for column in ["arrival_airport", "carrier_group", "is_night", "cargo"]:
    print(sample.groupby(column)["is_late"].agg(flights="count", late="sum").T, "\n")

arrival_airport  ATL  EWR  LAX  MIA  SFO
flights          305  465  287  399  270
late              35   56   40   64   55 

carrier_group  Alaska Airlines  American Airlines  Delta Air Lines  Other  \
flights                     93                298              339    512   
late                        21                 70               38     58   

carrier_group  United Airlines  
flights                    484  
late                        63   

is_night     0    1
flights   1589  137
late       238   12 

cargo       0    1
flights  1557  169
late      236   14 



### Check 3: batches per airport

How many batches each airport's result rests on, and how big they are.

In [6]:
batch_sizes = sample.groupby(["arrival_airport", "batch"]).size()
batch_sizes.groupby("arrival_airport").agg(batches="count", smallest="min", largest="max")

,batches,smallest,largest
arrival_airport,,,
ATL,11,22,30
EWR,11,39,45
LAX,11,22,29
MIA,11,30,43
SFO,11,21,26


### Check 4: no missing values in the model columns

In [7]:
MODEL_COLUMNS = ["is_late", "arrival_airport", "is_night", "carrier_group", "cargo", "weather_wind_speed"]
sample[MODEL_COLUMNS].isna().sum()

is_late               0
arrival_airport       0
is_night              0
carrier_group         0
cargo                 0
weather_wind_speed    0
dtype: int64

### Check 5: are any inputs too entangled to separate?

The variance inflation factor (VIF) measures how much an input overlaps with the others. Around
1 means independent. Above about 5 means the model will struggle to tell that input's effect
apart from the others. This uses the secondary model, which has more inputs to overlap.

In [8]:
design = dmatrix(INPUTS, sample, return_type="dataframe")

vif = pd.Series(
    [variance_inflation_factor(design.values, i) for i in range(design.shape[1])],
    index=design.columns,
).drop("Intercept")

vif.round(2).sort_values(ascending=False)

C(arrival_airport, Treatment("SFO"))[T.MIA]                            3.66
C(arrival_airport, Treatment("SFO"))[T.ATL]                            3.43
C(arrival_airport, Treatment("SFO"))[T.EWR]                            2.79
C(arrival_airport, Treatment("SFO"))[T.LAX]                            2.44
C(carrier_group, Treatment("United Airlines"))[T.Other]                2.09
weather_wind_speed                                                     2.03
C(carrier_group, Treatment("United Airlines"))[T.Delta Air Lines]      1.64
C(carrier_group, Treatment("United Airlines"))[T.American Airlines]    1.60
cargo                                                                  1.44
C(carrier_group, Treatment("United Airlines"))[T.Alaska Airlines]      1.28
is_night                                                               1.17
dtype: float64

### Check 6: how much overlap is there between airline and airport?

Share of each airline group's flights going to each airport.

In [9]:
pd.crosstab(sample["carrier_group"], sample["arrival_airport"], normalize="index").round(2)

arrival_airport,ATL,EWR,LAX,MIA,SFO
carrier_group,,,,,
Alaska Airlines,0.10,0.00,0.35,0.00,0.55
American Airlines,0.11,0.16,0.13,0.46,0.13
Delta Air Lines,0.47,0.16,0.09,0.15,0.13
Other,0.10,0.17,0.24,0.38,0.12
United Airlines,0.11,0.57,0.12,0.04,0.15


## Step 2 conclusion

_Snapshot from 2 October 2026. The checks above update on re-run; this text does not._

**The data can support the model as specified.**

- **Size:** 1,726 flights and 250 late, across 11 pulls, after the three filters (regular pulls,
  scheduled 2+ hours before the pull, scheduled operators). With 11 inputs that's about 23 late
  flights per input, comfortably above the rule of thumb of 10.
- **Every group has late flights**, so nothing will break the fit. The thinnest are night
  (12 late), cargo (14) and Alaska (21). Their own estimates will be imprecise, but they're
  controls, not the question.
- **No missing values** in any model column.
- **Overlap between inputs is acceptable:** every VIF is under 4. The airport inputs sit around
  3 because each is compared with SFO, the smallest airport group (270 flights).
- **Airline and airport overlap but can still be separated:** Alaska sends 55% of its flights to
  SFO and United 57% to EWR, yet every airline flies to at least three of the five airports.
- **Each airport rests on 11 batches of 21–45 flights.** That's the honest size of the evidence,
  and it's why every claim must also pass the cluster bootstrap.

**The specification above is now fixed.** Step 3 fits `PRIMARY_FORMULA` and `SECONDARY_FORMULA`
exactly as printed, runs the cluster bootstrap and checks R1 to R8, and reports all of them against
the rules in "What will count as a result". Then, around 21 October, the same code runs on pulls
from 3 October onward, as the true test.

# Step 3: Fit the model and read the results

Everything below follows the step 2 specification exactly. The specification was committed to git
(commit `de7f6ce`) before this section was written and before the specified models were fitted.
(Rough exploratory regressions were run while reviewing step 1, and they helped shape the
hypotheses. The "honest limit" in step 2 covers this: only the October test is a clean one.)

**How to read an odds ratio.** An odds ratio compares the odds of being late between two groups.
1 means no difference. Above 1 means more likely to be late, below 1 less likely. In the pairwise
table every airport is compared with SFO, so an odds ratio below 1 means *that airport* is late
less often than SFO.

**Added on 4 October 2026**, after the results below were first written up and before any October
data was examined: the safety checks in the next cell, the "October rule" section and the
sensitivity checks near the end. Each is labelled where it appears. None of them changes a
September result.

In [10]:
import warnings

import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import PerfectSeparationError

AIRPORTS = ["ATL", "EWR", "LAX", "MIA"]
AIRPORT_TERMS = {f'C(arrival_airport, Treatment("SFO"))[T.{airport}]': airport for airport in AIRPORTS}

## Which pulls this run uses

_Added 4 October 2026, before any October data was examined._ These checks run before anything is
fitted, so a stale or changed table stops the notebook instead of quietly giving different numbers.
The October test is a copy of this notebook in which only the two window dates in step 2 change
(see the amendment in the conclusion).

In [11]:
SEPTEMBER_RUN = (FIRST_REGULAR_PULL, LAST_REGULAR_PULL) == ("2026-09-07", "2026-10-01")

assert clean["pull_date"].max() >= pd.Timestamp(LAST_REGULAR_PULL), (
    "The data ends before the window does. Is this the old CSV backup, or an expired Snowflake trial?")

# The pipeline pulls on odd days of the month (cron "0 9 */2 * *"). Any other date means a manual
# or late-starting run, which the pull-date rule in flights_data.py can't place correctly.
assert (sample["pull_date"].dt.day % 2 == 1).all(), "A pull date is off the every-other-day schedule"

pulls = sorted(sample["pull_date"].dt.date.unique())
print(f"{len(pulls)} pulls, {pulls[0]} to {pulls[-1]}: {len(sample):,} flights, {sample['is_late'].sum()} late")
if SEPTEMBER_RUN:
    assert (len(pulls), len(sample), sample["is_late"].sum()) == (11, 1726, 250), "The September sample has changed"
else:
    assert len(pulls) >= 8, "Fewer than 8 of the 10 October pulls loaded: by the amendment, the test is not run"

# Operators that never appeared in September. Check that each is a scheduled airline (not cargo
# or on-demand, see flights_data.py) before trusting the fit.
september_carriers = set(clean.loc[clean["pull_date"] <= "2026-10-01", "carrier"])
new_carriers = sample.loc[~sample["carrier"].isin(september_carriers), "carrier"].value_counts()
print("Operators not seen in September:", new_carriers.to_dict() or "none")

11 pulls, 2026-09-07 to 2026-10-01: 1,726 flights, 250 late
Operators not seen in September: none


In [12]:
def fit_logit(formula, data, cluster_column="batch"):
    """Fit a logistic regression, with standard errors clustered by the given column.

    missing="raise" stops with an error if any model input is blank. Otherwise the formula would
    silently drop those flights, and two models could quietly fit different flights.
    """
    clusters = data[cluster_column].astype("category").cat.codes
    model = smf.logit(formula, data=data, missing="raise")
    return model.fit(cov_type="cluster", cov_kwds={"groups": clusters}, disp=False)


def odds_ratios(result, terms):
    """Odds ratio, 95% confidence interval and p-value for each term of a fitted model."""
    interval = result.conf_int().loc[terms]
    return pd.DataFrame({
        "odds ratio": np.exp(result.params[terms]),
        "ci low": np.exp(interval[0]),
        "ci high": np.exp(interval[1]),
        "p-value": result.pvalues[terms],
    })


def pairwise_table(result):
    """Each airport vs SFO from a fitted secondary model, with Holm-adjusted p-values."""
    table = odds_ratios(result, list(AIRPORT_TERMS)).rename(index=AIRPORT_TERMS)
    table["holm p-value"] = multipletests(table["p-value"], method="holm")[1]
    return table

## Primary test: SFO vs the other four airports together

In [13]:
primary = fit_logit(PRIMARY_FORMULA, sample)
primary_result = odds_ratios(primary, ["is_sfo"])
primary_result

,odds ratio,ci low,ci high,p-value
is_sfo,2.047,1.107,3.786,0.022


In [14]:
# The same result in percentage points: how much more likely an SFO flight is to be late,
# averaged over all flights, with everything else held constant.
# Columns by position: 0 = the effect, 4 and 5 = its 95% interval (statsmodels misspells one name).
effect = primary.get_margeff(at="overall", dummy=True).summary_frame().loc["is_sfo"]
print(f"SFO flights are {effect.iloc[0] * 100:+.1f} percentage points more likely to be late "
      f"(95% CI {effect.iloc[4] * 100:+.1f} to {effect.iloc[5] * 100:+.1f}), holding everything else constant.")

SFO flights are +10.1 percentage points more likely to be late (95% CI +0.2 to +20.1), holding everything else constant.


## Secondary tests: each airport vs SFO

Holm's adjustment raises the p-values to account for running four tests at once.

In [15]:
secondary = fit_logit(SECONDARY_FORMULA, sample)
pairwise = pairwise_table(secondary)
pairwise

,odds ratio,ci low,ci high,p-value,holm p-value
ATL,0.363,0.174,0.756,0.007,0.027
EWR,0.476,0.238,0.952,0.036,0.072
LAX,0.483,0.270,0.863,0.014,0.042
MIA,0.553,0.277,1.106,0.094,0.094


## Cluster bootstrap

Each airport's 11 batches are resampled with replacement, the model is re-fitted, and this is
repeated 999 times. The middle 95% of the results is the bootstrap confidence interval. It makes no
assumption about how many clusters there are, which is why every claim must pass it too.

In [16]:
BOOTSTRAP_REPS = 999


def fit_coefficients(formula, data, terms):
    """The coefficients for the given terms, or None if the model fails to fit.

    A resample counts as failed if either error is raised or the fit doesn't converge (newer
    statsmodels versions warn about perfect separation instead of raising, which shows up as
    non-convergence). If either model fails on a resample, both are skipped together.
    """
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result = smf.logit(formula, data=data, missing="raise").fit(disp=False)
    except (np.linalg.LinAlgError, PerfectSeparationError):
        return None
    if not result.mle_retvals["converged"]:
        return None
    return result.params[terms]


def run_bootstrap(data, primary_formula, secondary_formula, seed):
    """Re-fit both models on BOOTSTRAP_REPS resamples. Returns the draws and the failure count.

    Each resample draws each airport's batches with replacement. The batches are sorted first,
    so a fixed seed on the same data always gives the same draws, whatever order the rows are in.
    """
    rng = np.random.default_rng(seed)
    batches_by_airport = data.groupby("arrival_airport")["batch"].unique().apply(np.sort)
    rows_by_batch = dict(list(data.groupby("batch")))

    primary_draws, pairwise_draws, failed = [], [], 0
    for _ in range(BOOTSTRAP_REPS):
        picked = [
            rows_by_batch[batch]
            for batches in batches_by_airport
            for batch in rng.choice(batches, size=len(batches), replace=True)
        ]
        resample = pd.concat(picked, ignore_index=True)
        primary_coef = fit_coefficients(primary_formula, resample, ["is_sfo"])
        pairwise_coef = fit_coefficients(secondary_formula, resample, list(AIRPORT_TERMS))
        if primary_coef is None or pairwise_coef is None:
            failed += 1
            continue
        primary_draws.append(primary_coef)
        pairwise_draws.append(pairwise_coef)

    print(f"{failed} of {BOOTSTRAP_REPS} resamples failed to fit ({failed / BOOTSTRAP_REPS:.1%}). "
          f"The bootstrap counts as reliable below 5%.")
    return primary_draws, pairwise_draws, failed


primary_draws, pairwise_draws, failed = run_bootstrap(sample, PRIMARY_FORMULA, SECONDARY_FORMULA, seed=2026)

0 of 999 resamples failed to fit (0.0%). The bootstrap counts as reliable below 5%.


In [17]:
def bootstrap_interval(draws):
    """The middle 95% of the bootstrap odds ratios."""
    odds = np.exp(pd.DataFrame(draws))
    return odds.quantile([0.025, 0.975]).T.set_axis(["boot low", "boot high"], axis=1)


primary_summary = primary_result.join(bootstrap_interval(primary_draws))
pairwise_summary = pairwise.join(bootstrap_interval(pairwise_draws).rename(index=AIRPORT_TERMS))

## Verdicts, by the rules written in step 2

In [18]:
def primary_rule(row):
    """The step 2 rule: odds ratio above 1, and both 95% intervals (clustered, bootstrap) above 1."""
    return bool(row["odds ratio"] > 1 and row["ci low"] > 1 and row["boot low"] > 1)


primary_holds = primary_rule(primary_summary.loc["is_sfo"])
print("Primary test, SFO is later than the other four:", "YES" if primary_holds else "NO")
primary_summary

Primary test, SFO is later than the other four: YES


,odds ratio,ci low,ci high,p-value,boot low,boot high
is_sfo,2.047,1.107,3.786,0.022,1.149,3.724


In [19]:
def pairwise_verdict(row):
    """Apply the step 2 rule to one airport-vs-SFO comparison."""
    if row["odds ratio"] < 1 and row["holm p-value"] < 0.05 and row["boot high"] < 1:
        return "SFO is later"
    if row["odds ratio"] > 1 and row["holm p-value"] < 0.05 and row["boot low"] > 1:
        return "this airport is later than SFO"
    return "not distinguishable from SFO"


pairwise_summary["verdict"] = pairwise_summary.apply(pairwise_verdict, axis=1)
pairwise_summary

,odds ratio,ci low,ci high,p-value,holm p-value,boot low,boot high,verdict
ATL,0.363,0.174,0.756,0.007,0.027,0.175,0.724,SFO is later
EWR,0.476,0.238,0.952,0.036,0.072,0.234,0.896,not distinguishable from SFO
LAX,0.483,0.270,0.863,0.014,0.042,0.275,0.869,SFO is later
MIA,0.553,0.277,1.106,0.094,0.094,0.265,1.052,not distinguishable from SFO


## Robustness checks R1–R8

Each check re-fits both models with one change. R6 and R7 need flights outside the main sample, so
`model_table()` repeats step 2's column-building on any set of flights. The first line of the next
cell proves it reproduces the main sample exactly, so the checks use the same definitions.

In [20]:
def model_table(rows):
    """Add the model columns to a set of flights, exactly as step 2 built the main sample."""
    table = rows.copy()
    table["is_late"] = table["is_delayed_arrival"].astype(int)
    table["is_night"] = (table["arrival_hour"] < 6).astype(int)
    table["carrier_group"] = table["carrier"].where(table["carrier"].isin(MODEL_CARRIERS), "Other")
    table["cargo"] = table["is_cargo"].astype(int)
    table["is_sfo"] = table["is_sfo"].astype(int)
    table["batch"] = table["pull_date"].dt.strftime("%Y-%m-%d") + " " + table["arrival_airport"]
    return table


in_window = clean["pull_date"].between(FIRST_REGULAR_PULL, LAST_REGULAR_PULL)
scheduled_operator = ~clean["is_non_scheduled"]
early_enough = clean["hours_before_pull"] >= MIN_HOURS_BEFORE_PULL

rebuilt = model_table(clean[in_window & early_enough & scheduled_operator])
assert rebuilt.equals(sample), "model_table() no longer matches step 2"
print("model_table() reproduces the step 2 sample exactly.")

model_table() reproduces the step 2 sample exactly.


In [21]:
CONTROLS_NO_WIND = 'is_night + C(carrier_group, Treatment("United Airlines")) + cargo'
CONTROLS_NO_CARGO = 'is_night + C(carrier_group, Treatment("United Airlines")) + weather_wind_speed'

# R6: add the 4-5 September build-time pulls, without the 2-hour filter (those pulls didn't run at
# 09:00), clustered by departure date x airport because pull_date isn't meaningful for them. The
# pulls are named by date, so moving the window can never change what R6 adds. R6 is for September
# only: October has no build-time pulls, so the amendment drops it there.
MANUAL_PULL_DATES = ["2026-09-04", "2026-09-05"]
manual_pull = clean["pull_date"].isin(pd.to_datetime(MANUAL_PULL_DATES))
with_manual = model_table(clean[scheduled_operator & ((in_window & early_enough) | manual_pull)])
with_manual["date_batch"] = with_manual["flight_date"].dt.strftime("%Y-%m-%d") + " " + with_manual["arrival_airport"]
# R6 must add the build-time pulls and nothing else.
added = with_manual["pull_date"] < pd.Timestamp(FIRST_REGULAR_PULL)
assert added.sum() == (manual_pull & scheduled_operator).sum(), "R6 added flights from outside the build-time pulls"

# R7: keep the flights scheduled within 2 hours of the pull.
no_filter = model_table(clean[in_window & scheduled_operator])


def make_checks(controls, controls_no_cargo, main, r7_data):
    """Every check for one version of the main model, as {name: (controls, data, cluster column)}."""
    # R8: a 40-flight airline cut-off instead of 75.
    r8 = main.copy()
    carrier_counts = r8["carrier"].value_counts()
    r8["carrier_group"] = r8["carrier"].where(r8["carrier"].isin(carrier_counts[carrier_counts >= 40].index), "Other")

    checks = {
        "main": (controls, main, "batch"),
        "R1 no wind": (CONTROLS_NO_WIND, main, "batch"),
        "R2 + flight length": (controls + " + scheduled_duration_hours",
                               main.dropna(subset=["scheduled_duration_hours"]), "batch"),
        "R3 passenger only": (controls_no_cargo, main[main["cargo"] == 0], "batch"),
        "R4 + pull indicators": (controls + " + C(pull_date)", main, "batch"),
        "R6 + 4-5 Sep (no wind)": (CONTROLS_NO_WIND, with_manual, "date_batch"),
        "R7 no 2-hour filter": (controls, r7_data, "batch"),
        "R8 40-flight cut-off": (controls, r8, "batch"),
    }
    if not SEPTEMBER_RUN:
        del checks["R6 + 4-5 Sep (no wind)"]

    # Every check except R6 must stay inside the window.
    for name, (_, data, _) in checks.items():
        if not name.startswith("R6"):
            assert data["pull_date"].between(FIRST_REGULAR_PULL, LAST_REGULAR_PULL).all(), name
    return checks


CHECKS = make_checks(CONTROLS, CONTROLS_NO_CARGO, sample, no_filter)

In [22]:
def fit_check(controls, data, cluster_column):
    """Primary and pairwise odds ratios (with clustered CIs) for one version of the model."""
    primary_fit = fit_logit("is_late ~ is_sfo + " + controls, data, cluster_column)
    pairwise_fit = fit_logit('is_late ~ C(arrival_airport, Treatment("SFO")) + ' + controls, data, cluster_column)
    return pd.concat([
        odds_ratios(primary_fit, ["is_sfo"]).rename(index={"is_sfo": "SFO vs rest"}),
        odds_ratios(pairwise_fit, list(AIRPORT_TERMS)).rename(index=AIRPORT_TERMS),
    ])


check_results = {name: fit_check(*settings) for name, settings in CHECKS.items()}
flights_per_check = pd.Series({name: len(settings[1]) for name, settings in CHECKS.items()}, name="flights")
flights_per_check

main                      1726
R1 no wind                1726
R2 + flight length        1713
R3 passenger only         1557
R4 + pull indicators      1726
R6 + 4-5 Sep (no wind)    2463
R7 no 2-hour filter       1905
R8 40-flight cut-off      1726
Name: flights, dtype: int64

In [23]:
# Odds ratios across the checks. Rows: comparison. Columns: check.
pd.DataFrame({name: result["odds ratio"] for name, result in check_results.items()}).round(2)

,main,R1 no wind,R2 + flight length,R3 passenger only,R4 + pull indicators,R6 + 4-5 Sep (no wind),R7 no 2-hour filter,R8 40-flight cut-off
SFO vs rest,2.05,1.47,2.08,2.27,1.99,1.59,2.02,2.07
ATL,0.36,0.61,0.35,0.33,0.38,0.54,0.35,0.36
EWR,0.48,0.62,0.47,0.45,0.49,0.69,0.49,0.47
LAX,0.48,0.69,0.47,0.38,0.49,0.54,0.48,0.47
MIA,0.55,0.85,0.54,0.59,0.57,0.75,0.54,0.55


In [24]:
# Does each 95% interval exclude 1? (+1 = entirely above 1, -1 = entirely below, 0 = includes 1)
def interval_side(result):
    return np.where(result["ci low"] > 1, 1, np.where(result["ci high"] < 1, -1, 0))


pd.DataFrame(
    {name: interval_side(result) for name, result in check_results.items()},
    index=check_results["main"].index,
)

,main,R1 no wind,R2 + flight length,R3 passenger only,R4 + pull indicators,R6 + 4-5 Sep (no wind),R7 no 2-hour filter,R8 40-flight cut-off
SFO vs rest,1,0,1,1,1,1,1,1
ATL,-1,0,-1,-1,-1,-1,-1,-1
EWR,-1,0,-1,-1,-1,0,-1,-1
LAX,-1,0,-1,-1,-1,-1,-1,-1
MIA,0,0,0,0,0,0,0,0


In [25]:
# The primary comparison (SFO vs rest) in every check, with its 95% interval.
pd.DataFrame({name: result.loc["SFO vs rest"] for name, result in check_results.items()}).T.round(3)

,odds ratio,ci low,ci high,p-value
main,2.047,1.107,3.786,0.022
R1 no wind,1.470,0.942,2.292,0.089
R2 + flight length,2.075,1.117,3.854,0.021
R3 passenger only,2.272,1.216,4.243,0.010
R4 + pull indicators,1.991,1.170,3.391,0.011
R6 + 4-5 Sep (no wind),1.591,1.140,2.222,0.006
R7 no 2-hour filter,2.022,1.112,3.677,0.021
R8 40-flight cut-off,2.073,1.119,3.841,0.020


### R5: leave one pull out

The model is re-fitted 11 times, each time without one pull. If one evening were driving the
result, its odds ratio would jump when that pull is removed.

In [26]:
def leave_one_out_fits(controls, data):
    """Odds ratios re-fitted once per pull, each time without that pull. One column per pull."""
    results = {}
    for pull in sorted(data["pull_date"].unique()):
        without_pull = data[data["pull_date"] != pull]
        results[pd.Timestamp(pull).date()] = fit_check(controls, without_pull, "batch")["odds ratio"]
    return pd.DataFrame(results)


leave_one_out = leave_one_out_fits(CONTROLS, sample)
leave_one_out.agg(["min", "max"], axis=1).round(2)

,min,max
SFO vs rest,1.72,2.69
ATL,0.26,0.43
EWR,0.34,0.57
LAX,0.39,0.56
MIA,0.41,0.66


## Is each claim robust?

The step 2 rule: a claim is robust if its interval still excludes 1 in R1–R4 (clustered intervals),
and its direction holds in every leave-one-out fit (R5) and in R6–R8.

In [27]:
INTERVAL_CHECKS = ["R1 no wind", "R2 + flight length", "R3 passenger only", "R4 + pull indicators"]
DIRECTION_CHECKS = [name for name in ["R6 + 4-5 Sep (no wind)", "R7 no 2-hour filter", "R8 40-flight cut-off"]
                    if name in CHECKS]


def points_the_same_way(odds_ratio, direction):
    """True if an odds ratio is above 1 (direction +1) or below 1 (direction -1)."""
    return odds_ratio > 1 if direction == 1 else odds_ratio < 1


def is_robust(comparison, direction, check_results, leave_one_out):
    """direction is +1 if the claim is 'above 1' (SFO later overall), -1 if 'below 1'."""
    intervals_hold = all(
        interval_side(check_results[name].loc[[comparison]])[0] == direction for name in INTERVAL_CHECKS
    )
    directions_hold = all(
        points_the_same_way(check_results[name].loc[comparison, "odds ratio"], direction)
        for name in DIRECTION_CHECKS
    )
    leave_one_out_holds = bool(points_the_same_way(leave_one_out.loc[comparison], direction).all())
    return intervals_hold and directions_hold and leave_one_out_holds


def robust_claims(primary_holds, pairwise_summary, check_results, leave_one_out):
    """Apply the robustness rule to every claim the verdicts made."""
    claims = {}
    if primary_holds:
        claims["SFO vs rest"] = is_robust("SFO vs rest", +1, check_results, leave_one_out)
    for airport, verdict in pairwise_summary["verdict"].items():
        if verdict == "SFO is later":
            claims[airport] = is_robust(airport, -1, check_results, leave_one_out)
        elif verdict == "this airport is later than SFO":
            claims[airport] = is_robust(airport, +1, check_results, leave_one_out)
    return pd.Series(claims, name="robust", dtype=bool)


robust_claims(primary_holds, pairwise_summary, check_results, leave_one_out)

SFO vs rest    False
ATL            False
LAX            False
Name: robust, dtype: bool

## Why every claim fails R1: wind is measured after the delay

_This section was **not** in the step 2 plan. It was added after seeing that dropping wind (R1)
changes the answer, to understand why. Treat it as explanation, not as pre-registered evidence._

The main model's wind estimate points the wrong way: windier comes out *less* likely to be late.

In [28]:
wind = odds_ratios(primary, ["weather_wind_speed"])
print(f"Each extra mph of wind changes the odds of being late by a factor of "
      f"{wind['odds ratio'].iloc[0]:.3f} (95% CI {wind['ci low'].iloc[0]:.3f} to {wind['ci high'].iloc[0]:.3f})")

Each extra mph of wind changes the odds of being late by a factor of 0.963 (95% CI 0.927 to 1.002)


**The cause:** the pipeline attaches the weather reading taken just before each flight's
**actual** landing. A late flight lands later in the evening, when the wind has usually dropped,
so late flights pick up calmer readings. "Calm" ends up partly *caused by* being late. A control
measured after the outcome distorts everything else in the model, including SFO, the windiest
airport.

The fix is to take the reading at the **scheduled** arrival time, which is fixed before any delay.
(That holds for operators that publish a real schedule. For a few cargo operators the "scheduled"
arrival is really the landing time; see the sensitivity checks below.)

In [29]:
from flights_data import load_weather, wind_at_scheduled_arrival

weather = load_weather()
sample["wind_at_schedule"] = wind_at_scheduled_arrival(sample, weather)
print("Flights matched to a reading at their scheduled time:", sample["wind_at_schedule"].notna().sum(), "of", len(sample))
assert sample["wind_at_schedule"].notna().all(), "Some flights have no reading within 2 hours of their schedule (see the amendment)"

# How much later, calmer readings late flights picked up: wind at actual landing minus wind at schedule.
sample.assign(shift=sample["weather_wind_speed"] - sample["wind_at_schedule"]).pivot_table(
    index="is_sfo", columns="is_late", values="shift", aggfunc="mean"
).rename(index={0: "other airports", 1: "SFO"}, columns={0: "on time", 1: "late"}).round(2)

Loaded stg_weather live from Snowflake
Flights matched to a reading at their scheduled time: 1726 of 1726


is_late,on time,late
is_sfo,,
other airports,0.13,-0.12
SFO,0.22,-1.30


In [30]:
# The primary and secondary models again, with wind taken at the scheduled time.
retimed_controls = CONTROLS.replace("weather_wind_speed", "wind_at_schedule")
retimed = fit_check(retimed_controls, sample, "batch")

retimed_primary = fit_logit("is_late ~ is_sfo + " + retimed_controls, sample)
print("Wind, re-timed:", odds_ratios(retimed_primary, ["wind_at_schedule"]).round(3).to_dict("records")[0])
retimed_effect = retimed_primary.get_margeff(at="overall", dummy=True).summary_frame().loc["is_sfo"]
print(f"SFO, re-timed: {retimed_effect.iloc[0] * 100:+.1f} percentage points "
      f"(95% CI {retimed_effect.iloc[4] * 100:+.1f} to {retimed_effect.iloc[5] * 100:+.1f})")
retimed.round(3)

Wind, re-timed: {'odds ratio': 0.992, 'ci low': 0.959, 'ci high': 1.026, 'p-value': 0.632}
SFO, re-timed: +6.2 percentage points (95% CI -2.8 to +15.1)


,odds ratio,ci low,ci high,p-value
SFO vs rest,1.583,0.864,2.901,0.137
ATL,0.553,0.258,1.184,0.127
EWR,0.594,0.301,1.175,0.135
LAX,0.643,0.364,1.137,0.129
MIA,0.785,0.384,1.604,0.507


In [31]:
# For scale: the raw gap, with nothing held constant.
sample.groupby("is_sfo")["is_late"].mean().rename(index={0: "other airports", 1: "SFO"}).rename("late rate")

is_sfo
other airports    0.134
SFO               0.204
Name: late rate, dtype: float64

## The October rule, in code

_Added 4 October 2026, after the September results, and before any October data was examined._
The 2 October amendment made the model with wind at the **scheduled** arrival the October test's
main result, but only the original model had a verdict in code. This section applies step 2's rules
to the re-timed model, unchanged: the bootstrap, the primary rule, the Holm-adjusted pairwise rule,
and the robustness checks with wind taken at the scheduled time (R1 and R6 have no wind, so they
stay as they are). In October this is the verdict that counts.

**On the September data below it is post hoc.** It shows what the rule gives here, but it was written
after seeing these numbers, so it is not a pre-registered September result.

In [32]:
RETIMED_PRIMARY = "is_late ~ is_sfo + " + retimed_controls
RETIMED_SECONDARY = 'is_late ~ C(arrival_airport, Treatment("SFO")) + ' + retimed_controls

# Its own seed and its own random generator, so the step 2 bootstrap above is untouched.
retimed_primary_draws, retimed_pairwise_draws, retimed_failed = run_bootstrap(
    sample, RETIMED_PRIMARY, RETIMED_SECONDARY, seed=2027)

retimed_summary = odds_ratios(retimed_primary, ["is_sfo"]).join(bootstrap_interval(retimed_primary_draws))
retimed_holds = primary_rule(retimed_summary.loc["is_sfo"])
print("Amended primary test (wind at the scheduled arrival), SFO is later than the other four:",
      "YES" if retimed_holds else "NO")
retimed_summary

0 of 999 resamples failed to fit (0.0%). The bootstrap counts as reliable below 5%.
Amended primary test (wind at the scheduled arrival), SFO is later than the other four: NO


,odds ratio,ci low,ci high,p-value,boot low,boot high
is_sfo,1.583,0.864,2.901,0.137,0.881,2.828


In [33]:
retimed_pairwise = pairwise_table(fit_logit(RETIMED_SECONDARY, sample)).join(
    bootstrap_interval(retimed_pairwise_draws).rename(index=AIRPORT_TERMS))
retimed_pairwise["verdict"] = retimed_pairwise.apply(pairwise_verdict, axis=1)
retimed_pairwise

,odds ratio,ci low,ci high,p-value,holm p-value,boot low,boot high,verdict
ATL,0.553,0.258,1.184,0.127,0.509,0.261,1.209,not distinguishable from SFO
EWR,0.594,0.301,1.175,0.135,0.509,0.298,1.116,not distinguishable from SFO
LAX,0.643,0.364,1.137,0.129,0.509,0.377,1.160,not distinguishable from SFO
MIA,0.785,0.384,1.604,0.507,0.509,0.402,1.531,not distinguishable from SFO


In [34]:
# The robustness checks with wind at the scheduled time. R7's extra flights need the reading too.
no_filter["wind_at_schedule"] = wind_at_scheduled_arrival(no_filter, weather)
retimed_checks = make_checks(retimed_controls, CONTROLS_NO_CARGO.replace("weather_wind_speed", "wind_at_schedule"),
                             sample, no_filter)
retimed_check_results = {name: fit_check(*settings) for name, settings in retimed_checks.items()}
retimed_leave_one_out = leave_one_out_fits(retimed_controls, sample)

print("Robust claims under the amended rule:",
      robust_claims(retimed_holds, retimed_pairwise, retimed_check_results, retimed_leave_one_out).to_dict() or "none")
print("Leave-one-out range, SFO vs rest:", retimed_leave_one_out.loc["SFO vs rest"].agg(["min", "max"]).round(2).to_dict())
pd.DataFrame({name: result.loc["SFO vs rest"] for name, result in retimed_check_results.items()}).T.round(3)

Robust claims under the amended rule: none
Leave-one-out range, SFO vs rest: {'min': 1.31, 'max': 1.94}


,odds ratio,ci low,ci high,p-value
main,1.583,0.864,2.901,0.137
R1 no wind,1.470,0.942,2.292,0.089
R2 + flight length,1.588,0.858,2.938,0.141
R3 passenger only,1.666,0.909,3.054,0.099
R4 + pull indicators,1.510,0.923,2.468,0.101
R6 + 4-5 Sep (no wind),1.591,1.140,2.222,0.006
R7 no 2-hour filter,1.601,0.888,2.889,0.118
R8 40-flight cut-off,1.607,0.872,2.959,0.128


In [35]:
# Rough power of the October test: the chance its clustered interval clears 1 if SFO's true odds
# ratio is the September re-timed estimate. It assumes October's uncertainty matches September's,
# scaled for 10 pulls instead of 11. Requiring the bootstrap to clear 1 as well makes it lower.
row = retimed_summary.loc["is_sfo"]
standard_error = (np.log(row["ci high"]) - np.log(row["ci low"])) / (2 * 1.96) * np.sqrt(11 / 10)
power = 1 - norm.cdf(1.96 - np.log(row["odds ratio"]) / standard_error)
print(f"Approximate power with 10 pulls: {power:.0%}")

Approximate power with 10 pulls: 29%


## Sensitivity checks

_Post hoc, added 4 October 2026 after an audit of the data. None of these is a verdict input, in
September or October. They show how much the answer depends on things the pre-registered model
can't see. The October run reports them too, unchanged._

### 1. "Late" is measured at the runway, not the gate

AviationStack's actual arrival equals its runway (wheels-on) time in every raw record (9,500 of
9,500 since 4 September), and there is no gate-in time anywhere in the data. The schedule is a gate
time. So "late" here means **landing** 15+ minutes after the scheduled gate arrival, which is
more lenient than the 15-minute gate rule the government uses, and taxi-in time, which differs by
airport, is not controlled.

If SFO's taxi-in is k minutes longer than the other airports', a flight 15 minutes late at SFO's
gate is only 15 − k minutes late at the runway, so the fair SFO threshold is 15 − k. The first table
shifts SFO's threshold that way (wind at the scheduled arrival). The second moves every airport's
threshold together: 5–10 minutes at the runway is roughly what 15 minutes at the gate would mean.

In [36]:
def late_at(data, threshold, sfo_threshold=None):
    """The data with 'late' redefined as landing at least `threshold` minutes after schedule."""
    if sfo_threshold is None:
        sfo_threshold = threshold
    cut_off = np.where(data["is_sfo"] == 1, sfo_threshold, threshold)
    return data.assign(is_late=(data["arrival_delay_minutes"] >= cut_off).astype(int))


def sfo_effect(formula, data):
    """SFO vs rest: odds ratio, clustered 95% interval, and the average effect in percentage points."""
    fit = fit_logit(formula, data)
    row = odds_ratios(fit, ["is_sfo"]).loc["is_sfo"]
    points = fit.get_margeff(at="overall", dummy=True).summary_frame().loc["is_sfo"].iloc[0] * 100
    return {"late": data["is_late"].sum(), "odds ratio": row["odds ratio"],
            "ci low": row["ci low"], "ci high": row["ci high"], "points": points}


assert late_at(sample, 15)["is_late"].equals(sample["is_late"]), "15 minutes should reproduce the fitted outcome"

pd.DataFrame({
    f"SFO taxi-in {k:+d} min (SFO late at {15 - k}+)": sfo_effect(RETIMED_PRIMARY, late_at(sample, 15, 15 - k))
    for k in [-4, -2, 0, 2, 4]
}).T.round(3)

,late,odds ratio,ci low,ci high,points
SFO taxi-in -4 min (SFO late at 19+),243.0,1.323,0.738,2.373,3.535
SFO taxi-in -2 min (SFO late at 17+),246.0,1.424,0.782,2.593,4.575
SFO taxi-in +0 min (SFO late at 15+),250.0,1.583,0.864,2.901,6.157
SFO taxi-in +2 min (SFO late at 13+),255.0,1.755,0.961,3.203,7.803
SFO taxi-in +4 min (SFO late at 11+),258.0,1.871,1.015,3.450,8.875


In [37]:
thresholds = {}
for threshold in [5, 10, 15, 20]:
    data = late_at(sample, threshold)
    thresholds[(f"late at {threshold}+ min", "pre-registered (wind at landing)")] = sfo_effect(PRIMARY_FORMULA, data)
    thresholds[(f"late at {threshold}+ min", "wind at scheduled arrival")] = sfo_effect(RETIMED_PRIMARY, data)
pd.DataFrame(thresholds).T.round(3)

late  odds ratio  ci low  \
late at 5+ min  pre-registered (wind at landing)  380.0       1.680   0.935   
                wind at scheduled arrival         380.0       1.341   0.755   
late at 10+ min pre-registered (wind at landing)  305.0       1.907   1.029   
                wind at scheduled arrival         305.0       1.463   0.801   
late at 15+ min pre-registered (wind at landing)  250.0       2.047   1.107   
                wind at scheduled arrival         250.0       1.583   0.864   
late at 20+ min pre-registered (wind at landing)  194.0       2.267   1.260   
                wind at scheduled arrival         194.0       1.665   0.941   

                                                  ci high  points  
late at 5+ min  pre-registered (wind at landing)    3.020   9.443  
                wind at scheduled arrival           2.381   5.146  
late at 10+ min pre-registered (wind at landing)    3.532  10.348  
                wind at scheduled arrival           2.671   5.811  
late at 15+ min pre-registered (wind at landing)    3.786  10.123  
                wind at scheduled arrival           2.901   6.157  
late at 20+ min pre-registered (wind at landing)    4.080   9.843  
                wind at scheduled arrival           2.947   5.691

### 2. Operators whose schedule can't be trusted

Two problems found on 4 October 2026 (details in `flights_data.py`):
- For about 11 operators, mostly cargo, the source's "scheduled" arrival is really the **landing
  time**, so they almost never look late, whatever happened. That's why cargo looks punctual: it's
  a data artifact, not how cargo operates. It also means their time of day, 2-hour filter and "wind
  at the scheduled time" are all measured at landing.
- The keyword lists miss a few small cargo and on-demand operators.

SFO has almost none of these flights; they sit mostly at MIA, LAX and EWR.

In [38]:
from flights_data import ARRIVAL_SCHEDULE_IS_ACTUAL, MISSED_CARGO_WORDS, MISSED_NON_SCHEDULED_WORDS

schedule_is_landing = sample["carrier"].str.contains(ARRIVAL_SCHEDULE_IS_ACTUAL, case=False)
missed_cargo = sample["carrier"].str.contains(MISSED_CARGO_WORDS, case=False)
missed_non_scheduled = sample["carrier"].str.contains(MISSED_NON_SCHEDULED_WORDS, case=False)
print("Flights by operators whose 'schedule' is the landing time:", schedule_is_landing.sum(),
      sample.loc[schedule_is_landing, "arrival_airport"].value_counts().to_dict())
print("Their late rate:", round(sample.loc[schedule_is_landing, "is_late"].mean(), 3),
      "| other cargo:", round(sample.loc[~schedule_is_landing & (sample["cargo"] == 1), "is_late"].mean(), 3))

corrected_lists = sample.assign(cargo=(sample["cargo"].astype(bool) | missed_cargo).astype(int))[~missed_non_scheduled]
passenger_airlines = sample[(sample["cargo"] == 0) & ~schedule_is_landing & ~missed_cargo & ~missed_non_scheduled]


def odds_ratio_versions(data, remove=""):
    """SFO vs rest odds ratio in three versions of the model, optionally with one input removed."""
    formulas = {
        "pre-registered": PRIMARY_FORMULA,
        "wind at schedule": RETIMED_PRIMARY,
        "no wind (R1)": "is_late ~ is_sfo + " + CONTROLS_NO_WIND,
    }
    ratios = {name: np.exp(fit_logit(formula.replace(remove, ""), data).params["is_sfo"])
              for name, formula in formulas.items()}
    return pd.Series({"flights": len(data), **ratios})


pd.DataFrame({
    "as fitted": odds_ratio_versions(sample),
    "schedule-is-landing operators dropped": odds_ratio_versions(sample[~schedule_is_landing]),
    "corrected keyword lists": odds_ratio_versions(corrected_lists),
    "scheduled passenger airlines only": odds_ratio_versions(passenger_airlines, remove=" + cargo"),
}).T.round(3)

Flights by operators whose 'schedule' is the landing time: 110 {'MIA': 84, 'EWR': 15, 'LAX': 11}
Their late rate: 0.027 | other cargo: 0.157


,flights,pre-registered,wind at schedule,no wind (R1)
as fitted,1726.0,2.047,1.583,1.470
schedule-is-landing operators dropped,1616.0,2.130,1.595,1.456
corrected keyword lists,1716.0,2.028,1.571,1.465
scheduled passenger airlines only,1536.0,2.247,1.654,1.477


In [39]:
# Each airport's late rate as fitted, and for scheduled passenger airlines only.
pd.DataFrame({
    "as fitted": sample.groupby("arrival_airport")["is_late"].mean(),
    "scheduled passenger airlines only": passenger_airlines.groupby("arrival_airport")["is_late"].mean(),
}).round(3)

,as fitted,scheduled passenger airlines only
arrival_airport,,
ATL,0.115,0.118
EWR,0.120,0.124
LAX,0.139,0.126
MIA,0.160,0.213
SFO,0.204,0.205


### 3. The 56-minute ceiling

No arrival in the data is more than 56 minutes late. If that were the pull timing (late flights
still in the air at 09:00 UTC), flights scheduled long before the pull would show bigger delays.
They don't: the maximum is the same whether a flight was due 4 or 12 hours before the pull. The
second table shows how close each airport's late flights get to the ceiling.

In [40]:
sample.groupby(pd.cut(sample["hours_before_pull"], [2, 4, 6, 8, 12, 24]), observed=True)[
    "arrival_delay_minutes"].agg(flights="count", largest="max")

,flights,largest
hours_before_pull,,
"(2, 4]",308,53
"(4, 6]",477,55
"(6, 8]",507,55
"(8, 12]",405,56
"(12, 24]",28,48


In [41]:
late_flights = sample[sample["is_late"] == 1]
late_flights.groupby("arrival_airport")["arrival_delay_minutes"].agg(
    late="count", late_by_45_min_or_more=lambda minutes: (minutes >= 45).sum(), largest="max")

,late,late_by_45_min_or_more,largest
arrival_airport,,,
ATL,35,3,55
EWR,56,1,47
LAX,40,1,45
MIA,64,9,56
SFO,55,10,52


## Step 3 conclusion

_Snapshot from 2 October 2026 (1,726 flights, 250 late). Corrections and additions made on
4 October 2026 are marked; all of them are post hoc. The tables above update on re-run; this text
does not._

### In one sentence

**SFO flights land late more often than flights into the other four airports in the raw data (a 7
percentage-point gap), but once wind is measured correctly the controlled estimate is about 6 points
and not distinguishable from zero. One month of data isn't enough to say more. The pre-registered
model's stronger result is inflated by a flaw in how wind was measured.**

### The verdict, by the rules fixed in step 2

| Test | Result | Verdict |
|---|---|---|
| **Primary:** SFO vs the other four | Odds ratio **2.05** (95% CI 1.11–3.79; bootstrap 1.15–3.72), about 10 percentage points | **SFO is later** |
| ATL vs SFO | 0.36, Holm p = 0.03, bootstrap 0.18–0.72 | **SFO is later** |
| LAX vs SFO | 0.48, Holm p = 0.04, bootstrap 0.28–0.87 | **SFO is later** (borderline, see below) |
| EWR vs SFO | 0.48, Holm p = 0.07 | Not distinguishable |
| MIA vs SFO | 0.55, Holm p = 0.09 | Not distinguishable |
| **Robust?** | Every claim fails one check, R1 (dropping wind) | **No** |

These verdicts are reported exactly as the rules require. The bootstrap was reliable: none of the
999 resamples failed. The LAX verdict is borderline on a detail the plan didn't fix. With a
slightly more cautious way of computing p-values (a t distribution with 54 degrees of freedom
instead of the default normal), LAX's Holm p becomes 0.052, which would make it not distinguishable.

_Added 4 October 2026 (post hoc):_ the primary pass also depends on "late" being measured at the
runway (see the limits below). At a 5-minute threshold, roughly what 15 minutes at the gate means
at the runway, the pre-registered model gives 1.68 with a clustered interval of 0.94–3.02, which
includes 1. With wind at the scheduled arrival, the effect stays at about 5 to 6 percentage points
at every threshold from 5 to 20 minutes.

### Why the result isn't robust, and why that matters

Every check except one leaves SFO clearly later: R2 2.08, R3 2.27, R4 1.99, R7 2.02, R8 2.07.
Leaving out each pull in turn (R5), the result stays between 1.72 and 2.69. **The exception is
R1: without wind, SFO vs rest falls to 1.47 (95% CI 0.94–2.29).**

The after-the-fact investigation above found the reason. **Wind was measured after the delay
happened.** The pipeline attaches the weather reading taken just before each flight's *actual*
landing. A late flight lands later in the evening, when the wind has usually dropped, so late SFO
flights picked up readings 1.3 mph calmer than at their scheduled time, while on-time ones didn't.
That made "calm" look like a cause of lateness, and since SFO is the windiest airport, it inflated
SFO's estimate.

Taking wind at the **scheduled** arrival instead:
- The wind effect disappears: 0.99 per mph, p = 0.63.
- SFO vs rest becomes **1.58 (95% CI 0.86–2.90)**, about **6 percentage points** (−3 to +15).
- No airport is clearly different from SFO.

So the estimates without the wind flaw are around **1.5 to 1.6**:

| Version | SFO vs rest | 95% interval |
|---|---|---|
| Wind at scheduled time | 1.58 | 0.86–2.90 |
| No wind (R1) | 1.47 | 0.94–2.29 |
| No wind, plus the early pulls (R6, direction check only) | 1.59 | 1.14–2.22 |

All three point the same way. R6's interval excludes 1, but step 2 made R6 a direction check only,
and _(corrected 4 October 2026)_ it isn't evidence on its own: it has no wind control, its roughly 740
extra flights come from the 4–5 Sep pulls the main sample deliberately leaves out, those flights skip the
2-hour filter, and no bootstrap was run for it. The two versions that can be read for certainty
both include 1. With nothing held constant, SFO is late 20.4% of the time vs 13.4%, a gap of 7 points.

_Added 4 October 2026 (post hoc):_ applying the October rule to September, the model with wind at
the scheduled arrival gives **NO**: 1.58, clustered 0.86–2.90, bootstrap 0.88–2.83. No pairwise
comparison is distinguishable, and leaving out each pull moves it between 1.31 and 1.94.

### Against the expectation written in step 2

- **Correct:** SFO later than the others overall (in the pre-registered model), and not distinguishable from MIA.
- **Wrong:** SFO was expected to be clearly later than EWR and borderline against LAX. In the
  pre-registered model it came out the other way round. In the corrected versions, neither is clear.

### What this means for the October test

The step 2 plan used wind measured at actual landing, which this analysis has shown to be flawed.
**Amendment, made on 2 October 2026 before any October data exists:** the October test will
report the primary model with wind taken at the **scheduled** arrival as its main result, alongside
the original version and R1. All other rules stay as written. If SFO is clearly later in October
with wind measured correctly, the finding is confirmed.

**Details of the amendment, fixed on 4 October 2026.** The 3 October pull had reached the warehouse
by then, but no October flight had been examined: the end date added to step 2 keeps every run of
this notebook at the 1 October pull.

1. **Window:** pulls from **3 October to 21 October inclusive**, the 10 pulls of the every-other-day
   schedule. Later pulls are never included, whenever the test is run.
2. **How it runs:** a copy of this notebook (`03_october.ipynb`) in which only `FIRST_REGULAR_PULL`
   and `LAST_REGULAR_PULL` in step 2 change, to `"2026-10-03"` and `"2026-10-21"`. It runs once,
   after the 21 October load is confirmed, and the first complete run is the result. It needs live
   Snowflake or a newer export; the 2 October CSV backup ends on 1 October, and the window check
   stops the run if the data doesn't reach 21 October.
3. **Enough data:** at least 8 of the 10 pulls must load. With fewer, the test isn't run and is
   reported as not run.
4. **"Clearly later"** means the step 2 primary rule applied to the model with wind at the scheduled
   arrival: odds ratio above 1, and both 95% intervals (batch-clustered, and the 999-rep stratified
   cluster bootstrap) above 1. The pairwise rule (Holm) and the robustness rule are step 2's,
   applied to the same model: R1–R4, R5, R7 and R8 with wind at the scheduled time. **R6 is
   dropped**, because October has no build-time pulls. This is the "October rule" section above.
5. **Missing wind:** if the notebook stops because a flight has no wind reading (none within
   2 hours of the scheduled or actual arrival), those flights are dropped from the sample for every
   model (main, R1, the checks and the bootstrap), and the number dropped is reported. If more than
   5% are dropped, the result is reported as unreliable.
6. **Unchanged:** the 15-minute outcome, the operator keyword lists and the fixed airline list.
   Operators not seen in September are listed before the fit, for the record only. The sensitivity
   checks (runway thresholds, operators with untrustworthy schedules) are reported, but they are
   not verdict inputs.
7. **Power, and what each outcome means.** If SFO's true odds ratio is about 1.58, 10 pulls give
   roughly a **29% chance** of the interval clearing 1 (computed above), so October will most likely
   be inconclusive even if the effect is real. About 40 pulls (around 80 days) would be needed for
   80%. Fixed in advance: **clearly later** = confirmed. **Same direction but an interval including 1**
   = not confirmed at this sample size, which is *not* a refutation; the September estimate stays
   suggestive. **Odds ratio at or below 1** = evidence against.

A better long-term fix belongs in the pipeline: `fct_flight_events` should also carry the weather at
the scheduled arrival, so every future analysis gets the correctly timed reading by default.

### Corrections to the step 2 wording (added 4 October 2026)

Step 2 is frozen, so its wording is corrected here rather than edited:

- **Decision 1** calls the outcome "the standard airline and government definition of punctuality".
  The source's actual arrival is the **runway (wheels-on)** time in every record, while the
  schedule is a gate time. So "late" means *landing* 15+ minutes after the scheduled gate arrival:
  the same 15-minute convention, but more lenient than the government's gate-arrival rule and not
  directly comparable to official on-time figures. The −12 minute median is a runway figure too.
  "Capped at 56 min" is also the wrong word: no value is clipped (there's no pile-up at 56).
  Flights more than about 50–56 minutes late are simply missing.
- **Decision 2:** the 11 regular pulls ran every other day, with none on 27 and 29 September (the
  API quota guard skipped them). The scheduled 09:00 pull on 5 September is excluded along with the
  build-time pulls, because pull dates can't tell them apart.
- **Decision 3** says "the schedule is set in advance". That's true for airlines, but for about 11
  operators (mostly cargo, 110 flights in this sample: MIA 84, EWR 15, LAX 11) the source's
  "scheduled" arrival is really the landing time.
- **Decision 8** says "cargo is rarely late". That is mostly the same artifact: flights by those
  operators are late 2.7% of the time, other cargo 15.7%. The cargo input works as a data-quality
  control, not as a real cargo effect.

### Limits that still apply

- **The 56-minute ceiling.** Rows thin out from about 50 minutes late and stop at 56; they are
  missing, not clipped. It isn't the pull timing: the largest delay is 55–56 minutes whether a
  flight was due 4 or 12 hours before the pull. The likely cause is the source leaving out flights
  more than about an hour late (unconfirmed). This is selection on the outcome, so every late rate
  is understated, and unevenly: 10 of SFO's 55 late flights are 45+ minutes late, against 9 of 64 at
  MIA, 3 of 35 at ATL, 1 of 56 at EWR and 1 of 40 at LAX. So the missing flights more plausibly
  hide part of SFO's gap than create it, but the counts are too small to put a number on it.
- **Runway, not gate.** Taxi-in time differs by airport and isn't in the data. If SFO's taxi-in is
  4 minutes shorter than the others', the corrected effect falls to about 3.5 points (OR 1.32); if
  it is 4 minutes longer, it rises to about 9 points (OR 1.87, its interval just clearing 1). That's
  sensitivity check 1. The same-airline comparisons in step 1 share this exposure, so they don't
  confirm the airport effect independently.
- **Operators whose schedule is the landing time.** They make MIA look more punctual than it is:
  MIA is late 16.0% of the time as fitted, 21.3% for scheduled passenger airlines only (SFO 20.5%).
  The SFO estimate barely moves without them (wind at schedule 1.58 → 1.60, or 1.65 for
  scheduled passenger airlines only).
- **One month, 11 pulls:** each airport's result rests on 11 batches, so intervals are wide.
- **Same-data hypotheses:** the September result can suggest, but only new data can confirm.
